# pix2pix Training — Real Face to Comic

This notebook trains a **pix2pix** conditional GAN to translate real face images into comic-style images.


In [1]:
import os
import csv
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Random seed: {SEED}")

# Keep workers at 0 on Windows notebooks so memory-mapped arrays are read in this process.
NUM_WORKERS = 0
print(f"Number of workers: {NUM_WORKERS}")

Random seed: 42
Number of workers: 0


In [2]:
# ── Device ──
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")

Device: cuda


## 1. Setup & Load Data


In [3]:
# Use memory mapping so the large .npy files stay on disk and only batches are read into RAM.
train_real = np.load("../data/npy/train_real.npy", mmap_mode="r")
train_comic = np.load("../data/npy/train_comic.npy", mmap_mode="r")

val_real = np.load("../data/npy/val_real.npy", mmap_mode="r")
val_comic = np.load("../data/npy/val_comic.npy", mmap_mode="r")

In [4]:
# check data loading
print(f"Train - real: {train_real.shape}, comic: {train_comic.shape}")
print(f"Val   - real: {val_real.shape},   comic: {val_comic.shape}")

sample = train_real[0]
print(f"Sample pixel range:  [{sample.min():.1f}, {sample.max():.1f}]")

Train - real: (8000, 3, 256, 256), comic: (8000, 3, 256, 256)
Val   - real: (2000, 3, 256, 256),   comic: (2000, 3, 256, 256)
Sample pixel range:  [-1.0, 1.0]


## 2. Dataset & DataLoader

Wrap the NumPy arrays in a PyTorch `Dataset`. Data is already resized and normalized — the dataset just converts to tensors.


In [5]:
class FaceComicDataset(Dataset):
    """Reads paired samples lazily from memory-mapped NumPy arrays."""

    def __init__(self, real_array, comic_array):
        if len(real_array) != len(comic_array):
            raise ValueError("real_array and comic_array must have the same length")
        self.real = real_array
        self.comic = comic_array

    def __len__(self):
        return len(self.real)

    @staticmethod
    def _sample_to_tensor(array, idx):
        if isinstance(array, torch.Tensor):
            return array[idx]
        return torch.from_numpy(np.array(array[idx], copy=True))

    def __getitem__(self, idx):
        return self._sample_to_tensor(self.real, idx), self._sample_to_tensor(self.comic, idx)


train_dataset = FaceComicDataset(train_real, train_comic)
val_dataset = FaceComicDataset(val_real, val_comic)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Train:  {len(train_dataset):,} pairs  ({len(train_loader)} batches)")
print(f"Val:    {len(val_dataset):,} pairs  ({len(val_loader)} batches)")


Train:  8,000 pairs  (500 batches)
Val:    2,000 pairs  (125 batches)


## 3. Generator — U-Net

Encoder-decoder with skip connections. The encoder downsamples to a bottleneck, the decoder upsamples back, and skip connections carry high-frequency detail across.


In [6]:
class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class Generator(nn.Module):
    """U-Net generator sized for 256x256 input (down to 4x4 bottleneck)."""

    def __init__(self, in_ch=3, out_ch=3):
        super().__init__()
        # Encoder: 256->128->64->32->16->8->4
        self.d1 = UNetDown(in_ch, 64, normalize=False)   # 256->128
        self.d2 = UNetDown(64, 128)                      # 128->64
        self.d3 = UNetDown(128, 256)                     # 64->32
        self.d4 = UNetDown(256, 512)                     # 32->16
        self.d5 = UNetDown(512, 512)                     # 16->8
        self.d6 = UNetDown(512, 512)                     # 8->4 (bottleneck)

        # Decoder: 4->8->16->32->64->128->256
        self.u1 = UNetUp(512, 512, dropout=True)         # 4->8,   cat d5 -> 1024
        self.u2 = UNetUp(1024, 512, dropout=True)        # 8->16,  cat d4 -> 1024
        self.u3 = UNetUp(1024, 256)                      # 16->32, cat d3 -> 512
        self.u4 = UNetUp(512, 128)                       # 32->64, cat d2 -> 256
        self.u5 = UNetUp(256, 64)                        # 64->128, cat d1 -> 128

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_ch, 4, 2, 1),   # 128->256
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)

        u1 = self.u1(d6, d5)
        u2 = self.u2(u1, d4)
        u3 = self.u3(u2, d3)
        u4 = self.u4(u3, d2)
        u5 = self.u5(u4, d1)

        return self.final(u5)


## 4. Discriminator — PatchGAN

Classifies overlapping 70x70 patches as real or fake. This forces the generator to produce sharp, locally realistic textures everywhere in the image.


In [7]:
class Discriminator(nn.Module):
    def __init__(self, in_ch=6):
        super().__init__()
        def block(in_c, out_c, normalize=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_ch, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, real_input, target_or_fake):
        x = torch.cat([real_input, target_or_fake], dim=1)
        return self.model(x)


## 5. Checkpoint Utilities

In [8]:
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def save_checkpoint(gen, disc, opt_gen, opt_disc, epoch, metrics, checkpoint_dir, name="pix2pix"):
    """Save a full training checkpoint (models + optimizers + metadata)."""
    checkpoint = {
        "epoch": epoch,
        "gen_state_dict": gen.state_dict(),
        "disc_state_dict": disc.state_dict(),
        "opt_gen_state_dict": opt_gen.state_dict(),
        "opt_disc_state_dict": opt_disc.state_dict(),
        "metrics": metrics,
    }
    path = checkpoint_dir / f"{name}_epoch_{epoch:03d}.pt"
    torch.save(checkpoint, path)
    return path


def load_checkpoint(checkpoint_path, gen, disc, opt_gen=None, opt_disc=None):
    """Restore model (and optionally optimizer) state from a checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    gen.load_state_dict(ckpt["gen_state_dict"])
    disc.load_state_dict(ckpt["disc_state_dict"])
    if opt_gen is not None:
        opt_gen.load_state_dict(ckpt["opt_gen_state_dict"])
    if opt_disc is not None:
        opt_disc.load_state_dict(ckpt["opt_disc_state_dict"])
    return ckpt["epoch"], ckpt.get("metrics", {})


print(f"Checkpoint directory: {CHECKPOINT_DIR.resolve()}")

Checkpoint directory: /projects/e32706/gwr4170/checkpoints


## 6. Weight Initialization & Grid Search

We search over a grid of learning rates, batch sizes, and L1 penalty weights. Each configuration trains for a short number of epochs, and we pick the one with the lowest **validation L1 reconstruction loss**.


In [9]:
# ── Grid search hyperparameter space ──
GRID_LRS = [1e-4, 2e-4, 5e-4]
GRID_BATCH_SIZES = [8, 16, 32, 64]
GRID_LAMBDA_L1 = [50, 100, 150]
GRID_EPOCHS = 20
GRID_BETAS = (0.5, 0.999)

total_configs = len(GRID_LRS) * len(GRID_BATCH_SIZES) * len(GRID_LAMBDA_L1)
print(f"Grid search: {total_configs} configurations, {GRID_EPOCHS} epochs each")

Grid search: 36 configurations, 20 epochs each


In [10]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


def make_models(lr, betas):
    """Create fresh Generator + Discriminator + optimizers."""
    gen = Generator().to(DEVICE)
    disc = Discriminator().to(DEVICE)
    gen.apply(init_weights)
    disc.apply(init_weights)
    opt_gen = optim.Adam(gen.parameters(), lr=lr, betas=betas)
    opt_disc = optim.Adam(disc.parameters(), lr=lr, betas=betas)
    return gen, disc, opt_gen, opt_disc


_g = Generator()
gen_params = sum(p.numel() for p in _g.parameters())
_d = Discriminator()
disc_params = sum(p.numel() for p in _d.parameters())
del _g, _d
print(f"Generator params:     {gen_params:,}")
print(f"Discriminator params: {disc_params:,}")

Generator params:     29,238,275
Discriminator params: 2,768,641


In [15]:
def train_config(config_idx, total_configs, lr, batch_size, lambda_l1, num_epochs, train_dataset, val_dataset):
    """Train one grid-search configuration and return its final val L1 loss."""

    print(f"\n{'='*70}")
    print(f"Config {config_idx}/{total_configs} | LR={lr:.0e}, BS={batch_size}, L1={lambda_l1:.0f}")
    print(f"{'='*70}")

    loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    t_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kw)
    v_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kw)

    gen, disc, opt_gen, opt_disc = make_models(lr, GRID_BETAS)
    criterion_gan = nn.BCEWithLogitsLoss()
    criterion_l1 = nn.L1Loss()

    num_train_batches = len(t_loader)
    for epoch in range(1, num_epochs + 1):
        gen.train()
        disc.train()
        epoch_loss_d = 0.0
        epoch_loss_g = 0.0
        epoch_loss_g_gan = 0.0
        epoch_loss_g_l1 = 0.0

        for real_img, comic_img in t_loader:
            real_img, comic_img = real_img.to(DEVICE), comic_img.to(DEVICE)
            fake_img = gen(real_img)

            # ── Discriminator step ──
            pred_real = disc(real_img, comic_img)
            pred_fake = disc(real_img, fake_img.detach())
            loss_d = (
                criterion_gan(pred_real, torch.ones_like(pred_real) * 0.9)
                + criterion_gan(pred_fake, torch.zeros_like(pred_fake))
            ) / 2

            opt_disc.zero_grad()
            loss_d.backward()
            opt_disc.step()

            # ── Generator step ──
            pred_fake_for_g = disc(real_img, fake_img)
            loss_g_gan = criterion_gan(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
            loss_g_l1 = criterion_l1(fake_img, comic_img)
            loss_g = loss_g_gan + lambda_l1 * loss_g_l1

            opt_gen.zero_grad()
            loss_g.backward()
            opt_gen.step()

            epoch_loss_d += loss_d.item()
            epoch_loss_g += loss_g.item()
            epoch_loss_g_gan += loss_g_gan.item()
            epoch_loss_g_l1 += loss_g_l1.item()

        print(
            f"  Epoch {epoch}/{num_epochs} | "
            f"D: {epoch_loss_d / num_train_batches:.4f} | "
            f"G: {epoch_loss_g / num_train_batches:.4f} | "
            f"GAN: {epoch_loss_g_gan / num_train_batches:.4f} | "
            f"L1: {epoch_loss_g_l1 / num_train_batches:.4f}"
        )

    # ── Evaluate on validation set ──
    gen.eval()
    val_l1_sum, val_batches = 0.0, 0
    with torch.no_grad():
        for real_img, comic_img in v_loader:
            real_img, comic_img = real_img.to(DEVICE), comic_img.to(DEVICE)
            fake_img = gen(real_img)
            val_l1_sum += criterion_l1(fake_img, comic_img).item()
            val_batches += 1

    val_l1 = val_l1_sum / val_batches
    print(f"  Val L1: {val_l1:.6f}")

    del gen, disc, opt_gen, opt_disc
    torch.cuda.empty_cache()

    return val_l1


print("train_config() defined")

train_config() defined


In [16]:
LOG_DIR = Path("../logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
GRID_RESULTS_FILE = LOG_DIR / "grid_search_results.csv"

grid_fields = ["config_idx", "learning_rate", "batch_size", "lambda_l1", "val_l1"]
with open(GRID_RESULTS_FILE, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=grid_fields).writeheader()

grid_results = []
config_idx = 0

for lr in GRID_LRS:
    for bs in GRID_BATCH_SIZES:
        for l1 in GRID_LAMBDA_L1:
            config_idx += 1
            try:
                val_l1 = train_config(
                    config_idx=config_idx,
                    total_configs=total_configs,
                    lr=lr,
                    batch_size=bs,
                    lambda_l1=l1,
                    num_epochs=GRID_EPOCHS,
                    train_dataset=train_dataset,
                    val_dataset=val_dataset,
                )
            except RuntimeError as e:
                print(f"  SKIPPED (OOM or error): {e}")
                torch.cuda.empty_cache()
                continue

            grid_results.append({"config_idx": config_idx, "lr": lr, "bs": bs, "l1": l1, "val_l1": val_l1})

            with open(GRID_RESULTS_FILE, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=grid_fields)
                w.writerow({
                    "config_idx": config_idx,
                    "learning_rate": f"{lr:.0e}",
                    "batch_size": bs,
                    "lambda_l1": l1,
                    "val_l1": f"{val_l1:.6f}",
                })

print(f"\n{'='*70}")
print("Grid Search Complete!")
print(f"{'='*70}")

best_config = min(grid_results, key=lambda x: x["val_l1"])
print(f"\nBest Configuration (lowest val L1 = {best_config['val_l1']:.6f}):")
print(f"  Config #{best_config['config_idx']}")
print(f"  Learning Rate : {best_config['lr']:.0e}")
print(f"  Batch Size    : {best_config['bs']}")
print(f"  Lambda L1     : {best_config['l1']:.0f}")
print(f"\nAll results saved to {GRID_RESULTS_FILE}")


Config 1/36 | LR=1e-04, BS=8, L1=50
  Epoch 1/20 | D: 0.5505 | G: 13.7013 | GAN: 1.4018 | L1: 0.2460
  Epoch 2/20 | D: 0.5593 | G: 13.1836 | GAN: 1.3322 | L1: 0.2370
  Epoch 3/20 | D: 0.5416 | G: 12.9509 | GAN: 1.3493 | L1: 0.2320
  Epoch 4/20 | D: 0.5307 | G: 12.7283 | GAN: 1.3777 | L1: 0.2270
  Epoch 5/20 | D: 0.5225 | G: 12.5056 | GAN: 1.4046 | L1: 0.2220
  Epoch 6/20 | D: 0.5062 | G: 12.3427 | GAN: 1.4640 | L1: 0.2176
  Epoch 7/20 | D: 0.4923 | G: 12.2056 | GAN: 1.5277 | L1: 0.2136
  Epoch 8/20 | D: 0.4805 | G: 12.0706 | GAN: 1.5782 | L1: 0.2098
  Epoch 9/20 | D: 0.4899 | G: 11.8883 | GAN: 1.6086 | L1: 0.2056
  Epoch 10/20 | D: 0.4608 | G: 11.7972 | GAN: 1.6549 | L1: 0.2028
  Epoch 11/20 | D: 0.4589 | G: 11.7276 | GAN: 1.7183 | L1: 0.2002
  Epoch 12/20 | D: 0.4579 | G: 11.6864 | GAN: 1.7800 | L1: 0.1981
  Epoch 13/20 | D: 0.4434 | G: 11.5847 | GAN: 1.7737 | L1: 0.1962
  Epoch 14/20 | D: 0.4413 | G: 11.5923 | GAN: 1.8448 | L1: 0.1950
  Epoch 15/20 | D: 0.4316 | G: 11.5087 | GAN: 1.

KeyboardInterrupt: 

## 7. Full Training — Initialize with Best Config

In [ ]:
# ── Use best hyperparameters from grid search ──
BEST_LR = best_config["lr"]
BEST_BATCH_SIZE = best_config["bs"]
BEST_LAMBDA_L1 = best_config["l1"]
BETAS = (0.5, 0.999)
NUM_EPOCHS = 100
CHECKPOINT_EVERY = 10

print("Full training configuration:")
print(f"  Learning Rate : {BEST_LR:.0e}")
print(f"  Batch Size    : {BEST_BATCH_SIZE}")
print(f"  Lambda L1     : {BEST_LAMBDA_L1:.0f}")
print(f"  Epochs        : {NUM_EPOCHS}")
print(f"  Checkpoint every {CHECKPOINT_EVERY} epochs")

# ── Re-create DataLoaders with best batch size ──
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
train_loader = DataLoader(train_dataset, batch_size=BEST_BATCH_SIZE, shuffle=True, **loader_kw)
val_loader = DataLoader(val_dataset, batch_size=BEST_BATCH_SIZE, shuffle=False, **loader_kw)

# ── Fresh models + optimizers ──
gen, disc, opt_gen, opt_disc = make_models(BEST_LR, BETAS)
criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

gen_params = sum(p.numel() for p in gen.parameters())
disc_params = sum(p.numel() for p in disc.parameters())
print(f"\nGenerator params:     {gen_params:,}")
print(f"Discriminator params: {disc_params:,}")
print(f"Train: {len(train_dataset):,} pairs  ({len(train_loader)} batches)")
print(f"Val:   {len(val_dataset):,} pairs  ({len(val_loader)} batches)")

## 8. Training Loop

Losses are tracked separately for interpretability:
- **D Loss** — discriminator's BCE loss
- **G GAN Loss** — generator's adversarial (fool-the-discriminator) loss
- **G L1 Loss** — raw pixel-level reconstruction loss (unscaled)

Checkpoints are saved every N epochs and whenever the validation L1 improves.

In [ ]:
# Fix 4 val indices so we see the same samples every epoch
fixed_val_indices = [0, 50, 100, 150]
fixed_real = torch.stack([
    torch.from_numpy(np.array(val_real[i], copy=True))
    for i in fixed_val_indices
]).to(DEVICE)
fixed_comic = torch.stack([
    torch.from_numpy(np.array(val_comic[i], copy=True))
    for i in fixed_val_indices
]).to(DEVICE)


def denorm(x):
    """[-1, 1] -> [0, 1] for display."""
    return (x.cpu().detach() + 1) / 2


def show_samples(real, fake, target, epoch, save_path=None):
    """Show 4 samples: input | generated | ground truth."""
    fig, axes = plt.subplots(4, 3, figsize=(9, 12))
    for i in range(4):
        for j, (img, title) in enumerate(zip(
            [real[i], fake[i], target[i]],
            ["Input (Real)", "Generated", "Ground Truth"]
        )):
            axes[i][j].imshow(denorm(img).permute(1, 2, 0).clamp(0, 1))
            axes[i][j].axis("off")
            if i == 0:
                axes[i][j].set_title(title)
    fig.suptitle(f"Epoch {epoch}", fontsize=14)
    plt.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


# ── Logging setup ──
LOG_FILE = LOG_DIR / "pix2pix_training_log.csv"
SAMPLE_DIR = Path("../outputs/training_samples")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_EVERY = 10

history = {
    "d_loss_train": [],  "g_gan_train": [],  "g_l1_train": [],  "d_acc_train": [],  "g_acc_train": [],
    "d_loss_val": [],    "g_gan_val": [],    "g_l1_val": [],    "d_acc_val": [],    "g_acc_val": [],
}

log_fields = ["epoch", "d_loss_train", "g_gan_train", "g_l1_train", "d_acc_train", "g_acc_train",
              "d_loss_val", "g_gan_val", "g_l1_val", "d_acc_val", "g_acc_val"]
with open(LOG_FILE, "w", newline="") as f:
    csv.DictWriter(f, fieldnames=log_fields).writeheader()

best_val_l1 = float("inf")
best_model_path = None

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    gen.train()
    disc.train()

    d_loss_sum, g_gan_sum, g_l1_sum = 0.0, 0.0, 0.0
    d_correct, d_total = 0, 0
    g_correct, g_total = 0, 0

    for real_img, comic_img in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}", leave=False):
        real_img, comic_img = real_img.to(DEVICE), comic_img.to(DEVICE)
        fake_img = gen(real_img)

        # ── Discriminator ──
        pred_real = disc(real_img, comic_img)
        pred_fake = disc(real_img, fake_img.detach())
        loss_d = (
            criterion_gan(pred_real, torch.ones_like(pred_real) * 0.9)
            + criterion_gan(pred_fake, torch.zeros_like(pred_fake))
        ) / 2

        opt_disc.zero_grad()
        loss_d.backward()
        opt_disc.step()

        d_correct += (torch.sigmoid(pred_real) > 0.5).sum().item()
        d_correct += (torch.sigmoid(pred_fake) < 0.5).sum().item()
        d_total += pred_real.numel() + pred_fake.numel()

        # ── Generator ──
        pred_fake_for_g = disc(real_img, fake_img)
        g_correct += (torch.sigmoid(pred_fake_for_g.detach()) > 0.5).sum().item()
        g_total += pred_fake_for_g.numel()

        loss_g_gan = criterion_gan(pred_fake_for_g, torch.ones_like(pred_fake_for_g))
        loss_g_l1 = criterion_l1(fake_img, comic_img)
        loss_g = loss_g_gan + BEST_LAMBDA_L1 * loss_g_l1

        opt_gen.zero_grad()
        loss_g.backward()
        opt_gen.step()

        d_loss_sum += loss_d.item()
        g_gan_sum += loss_g_gan.item()
        g_l1_sum += loss_g_l1.item()

    n_train = len(train_loader)
    history["d_loss_train"].append(d_loss_sum / n_train)
    history["g_gan_train"].append(g_gan_sum / n_train)
    history["g_l1_train"].append(g_l1_sum / n_train)
    history["d_acc_train"].append(d_correct / d_total * 100)
    history["g_acc_train"].append(g_correct / g_total * 100)

    # ── Validation ──
    gen.eval()
    disc.eval()
    vd_loss, vg_gan, vg_l1 = 0.0, 0.0, 0.0
    vd_correct, vd_total = 0, 0
    vg_correct, vg_total = 0, 0

    with torch.no_grad():
        for real_img, comic_img in val_loader:
            real_img, comic_img = real_img.to(DEVICE), comic_img.to(DEVICE)
            fake_img = gen(real_img)

            pred_real = disc(real_img, comic_img)
            pred_fake = disc(real_img, fake_img)

            vd_loss += (
                criterion_gan(pred_real, torch.ones_like(pred_real))
                + criterion_gan(pred_fake, torch.zeros_like(pred_fake))
            ).item() / 2
            vg_gan += criterion_gan(pred_fake, torch.ones_like(pred_fake)).item()
            vg_l1 += criterion_l1(fake_img, comic_img).item()

            vd_correct += (torch.sigmoid(pred_real) > 0.5).sum().item()
            vd_correct += (torch.sigmoid(pred_fake) < 0.5).sum().item()
            vd_total += pred_real.numel() + pred_fake.numel()
            vg_correct += (torch.sigmoid(pred_fake) > 0.5).sum().item()
            vg_total += pred_fake.numel()

    n_val = len(val_loader)
    history["d_loss_val"].append(vd_loss / n_val)
    history["g_gan_val"].append(vg_gan / n_val)
    history["g_l1_val"].append(vg_l1 / n_val)
    history["d_acc_val"].append(vd_correct / vd_total * 100)
    history["g_acc_val"].append(vg_correct / vg_total * 100)

    elapsed = time.time() - t0

    # ── Print ──
    print(f"Epoch {epoch:03d}/{NUM_EPOCHS:03d}  ({elapsed:.0f}s)")
    print(
        f"  train | d_loss={history['d_loss_train'][-1]:.4f}  "
        f"g_gan={history['g_gan_train'][-1]:.4f}  "
        f"g_l1={history['g_l1_train'][-1]:.4f}  "
        f"d_acc={history['d_acc_train'][-1]:.1f}%  "
        f"g_acc={history['g_acc_train'][-1]:.1f}%"
    )
    print(
        f"  val   | d_loss={history['d_loss_val'][-1]:.4f}  "
        f"g_gan={history['g_gan_val'][-1]:.4f}  "
        f"g_l1={history['g_l1_val'][-1]:.4f}  "
        f"d_acc={history['d_acc_val'][-1]:.1f}%  "
        f"g_acc={history['g_acc_val'][-1]:.1f}%"
    )

    # ── CSV log ──
    with open(LOG_FILE, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=log_fields)
        w.writerow({
            "epoch": epoch,
            "d_loss_train": f"{history['d_loss_train'][-1]:.6f}",
            "g_gan_train": f"{history['g_gan_train'][-1]:.6f}",
            "g_l1_train": f"{history['g_l1_train'][-1]:.6f}",
            "d_acc_train": f"{history['d_acc_train'][-1]:.4f}",
            "g_acc_train": f"{history['g_acc_train'][-1]:.4f}",
            "d_loss_val": f"{history['d_loss_val'][-1]:.6f}",
            "g_gan_val": f"{history['g_gan_val'][-1]:.6f}",
            "g_l1_val": f"{history['g_l1_val'][-1]:.6f}",
            "d_acc_val": f"{history['d_acc_val'][-1]:.4f}",
            "g_acc_val": f"{history['g_acc_val'][-1]:.4f}",
        })

    # ── Periodic checkpoint ──
    if epoch % CHECKPOINT_EVERY == 0 or epoch == NUM_EPOCHS:
        metrics = {
            "d_loss": history["d_loss_val"][-1],
            "g_gan": history["g_gan_val"][-1],
            "g_l1": history["g_l1_val"][-1],
        }
        ckpt_path = save_checkpoint(gen, disc, opt_gen, opt_disc, epoch, metrics, CHECKPOINT_DIR)
        print(f"  checkpoint saved: {ckpt_path.name}")

    # ── Best model (lowest val L1) ──
    current_val_l1 = history["g_l1_val"][-1]
    if current_val_l1 < best_val_l1:
        best_val_l1 = current_val_l1
        metrics = {
            "d_loss": history["d_loss_val"][-1],
            "g_gan": history["g_gan_val"][-1],
            "g_l1": current_val_l1,
        }
        best_model_path = save_checkpoint(gen, disc, opt_gen, opt_disc, epoch, metrics, CHECKPOINT_DIR, name="pix2pix_best")
        print(f"  * new best val L1: {current_val_l1:.6f} -> {best_model_path.name}")

    # ── Visual samples ──
    if epoch % SAMPLE_EVERY == 0 or epoch == NUM_EPOCHS:
        with torch.no_grad():
            fixed_fake = gen(fixed_real)
        sample_path = SAMPLE_DIR / f"pix2pix_epoch_{epoch:03d}.png"
        show_samples(fixed_real, fixed_fake, fixed_comic, epoch, save_path=sample_path)
        print(f"  samples saved: {sample_path}")

print(f"\n{'='*70}")
print("Training complete!")
print(f"{'='*70}")
print(f"Best val L1: {best_val_l1:.6f}")
print(f"Best model : {best_model_path}")

## 9. Loss & Accuracy Curves

In [ ]:
epochs_range = range(1, len(history["d_loss_train"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plots = [
    ("d_loss_train", "d_loss_val", "Discriminator Loss"),
    ("g_gan_train", "g_gan_val", "Generator GAN Loss (adversarial)"),
    ("g_l1_train", "g_l1_val", "Generator L1 Loss (reconstruction)"),
]

for ax, (train_key, val_key, title) in zip(axes.flat[:3], plots):
    ax.plot(epochs_range, history[train_key], label="Train")
    ax.plot(epochs_range, history[val_key], label="Val")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

ax4 = axes.flat[3]
ax4.plot(epochs_range, history["d_acc_train"], label="D Acc (Train)")
ax4.plot(epochs_range, history["d_acc_val"], label="D Acc (Val)")
ax4.plot(epochs_range, history["g_acc_train"], label="G Fooling (Train)", linestyle="--")
ax4.plot(epochs_range, history["g_acc_val"], label="G Fooling (Val)", linestyle="--")
ax4.set_title("Accuracy / Fooling Rate (%)", fontsize=12, fontweight="bold")
ax4.set_xlabel("Epoch")
ax4.set_ylabel("%")
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Final Visual Samples

Side-by-side comparison on 8 validation images: **Input (Real) → Generated Comic → Ground Truth Comic**.

In [ ]:
sample_indices = list(range(min(8, len(val_real))))

sample_real = torch.stack([
    torch.from_numpy(np.array(val_real[i], copy=True))
    for i in sample_indices
]).to(DEVICE)
sample_comic = torch.stack([
    torch.from_numpy(np.array(val_comic[i], copy=True))
    for i in sample_indices
]).to(DEVICE)

gen.eval()
with torch.no_grad():
    sample_fake = gen(sample_real)

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(9, 3 * len(sample_indices)))
if len(sample_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(len(sample_indices)):
    for j, (img, title) in enumerate(zip(
        [sample_real[i], sample_fake[i], sample_comic[i]],
        ["Input (Real)", "Generated Comic", "Ground Truth Comic"]
    )):
        axes[i][j].imshow(denorm(img).permute(1, 2, 0).cpu().clamp(0, 1))
        axes[i][j].axis("off")
        if i == 0:
            axes[i][j].set_title(title)

plt.tight_layout()
plt.show()

## 11. Save Final Weights

Save a lightweight generator-only file for inference (no optimizer or discriminator overhead).

In [ ]:
gen_weights_path = CHECKPOINT_DIR / "generator_final.pt"
torch.save(gen.state_dict(), gen_weights_path)
print(f"Generator weights saved: {gen_weights_path}")
print(f"  Size: {gen_weights_path.stat().st_size / (1024 * 1024):.1f} MB")

print(f"\nAll artifacts:")
print(f"  Training log       : {LOG_FILE}")
print(f"  Grid search results: {GRID_RESULTS_FILE}")
print(f"  Best checkpoint    : {best_model_path}")
print(f"  Generator weights  : {gen_weights_path}")
print(f"  Checkpoint dir     : {CHECKPOINT_DIR}")

ckpts = sorted(CHECKPOINT_DIR.glob("*.pt"))
print(f"\nCheckpoints ({len(ckpts)} files):")
for c in ckpts:
    size_mb = c.stat().st_size / (1024 * 1024)
    print(f"  {c.name:40s} ({size_mb:.1f} MB)")